# M145 bulk data

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as numpy
import matplotlib.pyplot as plt
from tueplots import fonts, fontsizes, cycler

import matplotlib.pyplot as plt
import os
import sys
from jax import random
from jax import flatten_util
import jax.numpy as np
import jax


import pandas as pd
# Build an absolute path from this notebook's parent directory
module_path = os.path.abspath(os.path.join('..', 'src'))

# Add to sys.path if not already present
if module_path not in sys.path:
    sys.path.append(module_path)

# Load our modules
from utils import *
from tsne_jax import *
#import tikzplotlib

from tueplots.constants.color import palettes
plt.rcParams.update(cycler.cycler(color=palettes.tue_plot))

In [5]:
import numpy as onp
X = pd.read_csv('../data/M145/wt_featurecounts_log_qnorm.txt', header=0, sep='\t', index_col=0)

# Step 1: Group every 3 rows (replicates) together
replicate_groups = onp.repeat(onp.arange(8), 3)  # 8 groups of 3 replicates each

# Step 2: Compute mean per group
df_mean = X.groupby(replicate_groups).mean()

# Step 3: Compute variance within each group of 3 replicates
# This results in a numpy array of shape (8, number of columns)
df_vars = X.groupby(replicate_groups).var()

# Set the top x% threshold (e.g., top 50%)
top_percent = 5

# Calculate variance for each column
variances = df_mean.var()

# Determine the cutoff variance value
cutoff = variances.quantile(1 - top_percent / 100)

# Filter columns with variance in the top x%
X = df_mean.loc[:, variances >= cutoff]

df_vars = df_vars.loc[:, variances >=cutoff]
flatten_vars, _ = flatten_util.ravel_pytree(df_vars.values)

#N = np.identity(X.values.shape[0])
input_cov = flatten_vars


In [6]:
labels_unique = ['t1', 't2', 't3', 't4', 't5', 't6', 't7', 't8']
labels = labels_unique

### Compute t-SNE embedding

In [ ]:
from openTSNE import TSNE
X_array = X.values
key = random.PRNGKey(42)

y_guess = random.normal(key, shape=(X_array.shape[0], 2))
Y_star = tsne_fwd(X_array, y_guess)

X_flat, X_unflattener = flatten_util.ravel_pytree(np.array(X_array))   # row-wise
Y_flat, Y_unflattener = flatten_util.ravel_pytree(Y_star)

np.save('../data/M145/tsne_embedding.npy', Y_star)

Perplexity value 30 is too high. Using perplexity 2.33 instead


===> Finding 7 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 0.07 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 0.00 seconds
===> Running optimization with exaggeration=12.00, lr=0.67 for 250 iterations...
Iteration   50, KL divergence 0.6197, 50 iterations in 0.2961 sec
Iteration  100, KL divergence 0.6292, 50 iterations in 0.2473 sec
Iteration  150, KL divergence 0.6289, 50 iterations in 0.2475 sec
Iteration  200, KL divergence 0.6290, 50 iterations in 0.2459 sec
Iteration  250, KL divergence 0.6290, 50 iterations in 0.2465 sec
   --> Time elapsed: 1.28 seconds
===> Running optimization with exaggeration=1.00, lr=8.00 for 1000 iterations...
Iteration   50, KL divergence 0.0423, 50 iterations in 0.2451 sec
Iteration  100, KL divergence 0.0324, 50 iterations in 0.2457 sec
Iteration  150, KL divergence 0.0315, 50 iterations in 0.2454 sec
Iteration  200, KL divergence 0.0312, 50 iterations in 0.2442 sec
Iteration  250

### Sensitivity Analysis

In [ ]:
# Check that the t-SNE embedding is a minimum of the cost function (KL divergence). Expected result: all partial derivatives are (almost) zero
KL_divergence_dy(X_flat, Y_flat, X_unflattener, Y_unflattener, perplexity=2.33)

NameError: name 'KL_divergence_dy' is not defined

In [ ]:
# Compoute Jacobian of sensitivities
sensitivities = compute_sensitivities(X_flat, Y_flat, X_unflattener, Y_unflattener, 2.33)
np.save('../data/M145/sensitivities.npy', sensitivities)

In [10]:
sensitivities.shape

(16, 3168)

In [ ]:
dy_dx_per_input = np.sum(np.abs(sensitivities), axis=0)
dy_dx_per_input_reshaped = X_unflattener(dy_dx_per_input) + 1e-8
np.save('../data/M145/sensitivities_summarized.npy', dy_dx_per_input_reshaped)

In [ ]:
actinorhodin = X.values[:, np.arange(240, 262)]
np.save('../data/M145/actinorhodin.npy', actinorhodin)

### Error propagation

In [18]:
cov_final = compute_cov_without_kronecker(X_flat, Y_flat, X_unflattener, Y_unflattener, np.diag(input_cov), 2.33)
cov_final = cov_final + 1e-3*np.eye(len(cov_final))

In [32]:
n_samples = 15
y_int = [i for i in range(Y_star.shape[0])]
S = equipotential_standard_normal(2 * Y_star.shape[0], n_samples)
L, lower = jax.scipy.linalg.cho_factor(cov_final, lower=True)
samples = np.transpose(np.transpose(np.dot(L, S))+Y_flat)

In [ ]:
np.save('..data/M145/samples_for_animation.npy', samples)